In [ ]:
#----------------------------------------------------------------
#      DATABASE AND GMAIL API CONNECTION TESTING SECTION
# --------------------------------------------------------------- 

# REQUIRED LIBRARIES #COPY
import os
import sys
import base64
import pandas as pd
import numpy as np
import textwrap
import requests
import pycountry
import psycopg2
from psycopg2.extras import execute_batch
import json
from dotenv import load_dotenv
from pathlib import Path
from cryptography.fernet import Fernet

# GOOGLE AUTH AND SERVICE LIBRARIES
from email.mime.text import MIMEText
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow

# PDF AUDIT REPORT LIBRARIES
from datetime import datetime

from email.mime.multipart import MIMEMultipart
from email.mime.application import MIMEApplication

from reportlab.lib import colors
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak)
from reportlab.lib.units import inch

# DATABASE CONNECTION
load_dotenv()

# DATABASE CREDENTIALS
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_PORT = os.getenv("DB_PORT")

IS_CI = os.getenv("GITHUB_ACTIONS", "false").lower() == "true"

SCOPES = ["https://www.googleapis.com/auth/gmail.send"]

# TOKEN ENCRYPTION AND SECURITY SECTION
ENCRYPTION_KEY = os.getenv("TOKEN_ENCRYPTION_KEY")

if not ENCRYPTION_KEY:
    raise ValueError("Missing TOKEN_ENCRYPTION_KEY")

fernet = Fernet(ENCRYPTION_KEY.encode())


def save_token(creds):

    encrypted_token = fernet.encrypt(
        creds.to_json().encode()
    )

    # LOCAL BASED ENV AND FILES USE, SECTION
    if not IS_CI:
        with open("token.enc", "wb") as f:
            f.write(encrypted_token)


def load_token():

    try:

        # GITHUB ACTIONS / CI SECTION
        if IS_CI:

            token_json = os.getenv("GMAIL_TOKEN_JSON")

            if not token_json:
                return None

            return Credentials.from_authorized_user_info(
                json.loads(token_json),
                SCOPES
            )

        # LOCAL ENVIRONMENT SECTION
        else:

            if not os.path.exists("token.enc"):
                return None

            with open("token.enc", "rb") as f:
                encrypted_token = f.read()

            decrypted_token = fernet.decrypt(
                encrypted_token
            )

            token_info = json.loads(
                decrypted_token.decode()
            )

            return Credentials.from_authorized_user_info(
                token_info,
                SCOPES
            )

    except Exception as e:

        if not IS_CI:
            print(f"Token load failed {e}")

        return None

# EMAIL SENDING FUNCTION
def send_email(service, subject, body, attachment_path=None):

    msg = MIMEMultipart()
    msg["To"] = TO_EMAIL
    msg["Subject"] = subject

    # EMAIL BODY
    msg.attach(MIMEText(textwrap.dedent(body), "plain"))

    # ATTACHMENT
    if attachment_path:
        with open(attachment_path, "rb") as f:
            attachment = MIMEApplication(f.read(), _subtype="pdf")
        attachment.add_header("Content-Disposition", "attachment", filename=os.path.basename(attachment_path))
        msg.attach(attachment)

    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()

    service.users().messages().send(
        userId="me",
        body={"raw": raw}
    ).execute()

TO_EMAIL = os.getenv("GMAIL_RECIPIENT")

if not TO_EMAIL:
    raise ValueError("Missing GMAIL_RECIPIENT")

creds = load_token()

# REFRESH IF POSSIBLE WITH 3 ATTEMPTS IN CASE OF FAILURE
if creds and creds.expired and creds.refresh_token:

    refresh_success = False

    for attempt in range(1, 4):

        try:
            creds.refresh(Request())
            if not IS_CI:
                save_token(creds)
            refresh_success = True
            break

        except Exception:
            continue

    if not refresh_success:

        creds = None

        if not IS_CI:
            print("Credential refresh attempts failed")

# IF NO CREDS, THEN RUN THE LOGIN POP-UP
if not creds:

    if IS_CI:
        raise RuntimeError(
            "Running in CI environment: Gmail token secret is invalid or missing"
        )

    flow = InstalledAppFlow.from_client_secrets_file(
        "gmail_credentials.json",
        SCOPES
    )

    creds = flow.run_local_server(port=0)

# SAVE TOKEN FOR LOCAL REUSE
if not creds:
    raise RuntimeError("No valid Gmail credentials available")

if not IS_CI:
    save_token(creds)

# GMAIL SERVICE INITIALIZATION
def get_gmail_service():

    try:
        service = build("gmail", "v1", credentials=creds)
        if not IS_CI:
            print("Gmail API service initialized successfully")
        return service 

    except Exception as e:
        if not IS_CI:
            print(f"Failed to initialize Gmail service: {e}")
        raise RuntimeError("Failed to initialize Gmail service")

service = get_gmail_service()

# TESTING THE CONNECTION
cur = None
connection = None

try:

    connection = psycopg2.connect(
        host=DB_HOST,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT
    )

    cur = connection.cursor()

    cur.execute("SELECT version();")
    engine_version = cur.fetchone()[0]

    cur.execute("SELECT CURRENT_DATABASE();")
    database_name = cur.fetchone()[0]

    cur.execute("SET search_path TO world_population;")
    cur.execute("SELECT CURRENT_SCHEMA();")
    schema_name = cur.fetchone()[0]

    # EMAIL CONTENT
    body = (f"""The database has been successfully connected.

Engine version:
{engine_version}

Database and schema name:
{database_name} | {schema_name}
"""
)

    #EMAIL NOTIFICATION SEND
    send_email(
        service,
        "SUCCESSFUL Database Connection - Gmail API",
        body
    )

    if not IS_CI:
        print("Success database connection, email has been sent")

except Exception as e:

    send_email(
        service,
        "FAILED Database Connection - Gmail API",
        str(e)
    )

    if not IS_CI:
        print("Failed database connection, email has been sent")

    raise RuntimeError("Database connection failed") from e

In [ ]:
#-------------------------------------------------------------------
# API DATA EXTRACTION, CLEANING AND ENRICHEMENT SECTION
#-------------------------------------------------------------------

# WORLD BANK APIs (POPULATION & FERTILITY RATE DATA)

indicator_map = {
    "population": "population_count",
    "fertility": "tfr"
}

apis = {
    "population":"https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&per_page=20000",
    "fertility":"https://api.worldbank.org/v2/country/all/indicator/SP.DYN.TFRT.IN?format=json&per_page=20000"
}

results = {}
cleaned_data = {}

api_debug = {}

try:
    for api_name, url in apis.items():
        current_api = api_name

        response = None
        
        for attempt in range(1, 4):
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                break

            except requests.RequestException:
                if attempt == 3:
                    if not IS_CI:
                        print("API failed, 3 attempts already completed")
                    raise

        api_debug[api_name] = {
            "url": url,
            "status_code": response.status_code,
            "headers": dict(response.headers),
            "response_preview": response.text[:450],
            "response_length": len(response.text)
        }

        try:
            json_data = response.json()
        except ValueError:
            raise ValueError("WORLD BANK API ERROR: Invalid JSON response (likely HTML, empty response, or malformed API request)")

        if not (
            isinstance(json_data, list)
            and len(json_data) > 1
            and isinstance(json_data[1], list)
    ):
            raise ValueError(f"Invalid structure for {api_name}")

        results[api_name] = json_data[1]

except Exception as api_issues:

    body = f"""ETL stopped due to API failure.

ERROR MESSAGE:
{api_issues}

FAILED API:
{current_api if 'current_api' in locals() else 'unknown'}

API DEBUG:
{api_debug.get(current_api, {"error": "No debug info available"})}

Response Length:
{api_debug.get(current_api, {}).get("response_length", "unknown")}
"""

    #EMAIL NOTIFICATION SEND
    send_email(
        service,
        "API - ISSUES",
        body
    )

    if not IS_CI:
        print("API ISSUES")
    sys.exit("ETL STOPPED")

# DATA CLEANING & TRANSFORMATION
def clean_world_bank_data(data, column_name):

    df = pd.json_normalize(data)

    df = df[["country.value", "countryiso3code", "date", "value"]]

    df = df.rename(columns={
    "country.value": "country_name",
    "countryiso3code": "country_code",
    "date": "year_record",
    "value": column_name
})

    inclusion = {"XKX", "CHI"}
    valid_country_codes = {country.alpha_3 for country in pycountry.countries}
    valid_country_codes = valid_country_codes.union(inclusion)

    df = df[df["country_code"].isin(valid_country_codes)]

    df["country_name"] = df["country_name"].astype("string").str.strip()
    df["country_code"] = df["country_code"].str.strip().str.upper()

    df["year_record"] = pd.to_numeric(df["year_record"], errors="coerce").astype("Int64")
    df[column_name] = pd.to_numeric(df[column_name], errors="coerce")

    return df

# CLEANING FOR BOTH APIs DATA REUSING THE FUNCTION "clean_world_bank_data".
for api_name, data in results.items():

    column_name = indicator_map[api_name]

    cleaned_data[api_name] = clean_world_bank_data(data, column_name)

population_df = cleaned_data["population"]
fertility_df = cleaned_data["fertility"]

country_names = {
"Venezuela, RB": "Venezuela",
"Iran, Islamic Rep.": "Iran",
"Korea, Rep.": "South Korea",
"Korea, Dem. People's Rep.": "North Korea",
"Egypt, Arab Rep.": "Egypt",
"Russian Federation": "Russia",
"Syrian Arab Republic": "Syria",
"Yemen, Rep.": "Yemen",
"Viet Nam": "Vietnam",
"Tanzania, United Republic of": "Tanzania",
"St. Martin (French part)": "French St. Martin",
"Sint Maarten (Dutch part)": "Dutch Sint Maarten",
"Puerto Rico (US)": "Puerto Rico",
"Micronesia, Fed. Sts.": "Micronesia",
"Moldova, Republic of": "Moldova",
"Bahamas, The": "Bahamas",
"Virgin Islands, British": "British Virgin Islands",
"Virgin Islands (U.S.)": "American Virgin Islands",
"West Bank and Gaza": "Palestine",
"Congo, Dem. Rep.": "Democratic Republic of the Congo",
"Congo, Rep.": "Republic of the Congo",
"Somalia, Fed. Rep.": "Somalia",
"Slovak Republic": "Republic of Slovakia",
"Gambia, The": "Gambia"
}

def rename_countries(country):
    if pd.isna(country):
        return country
    country = str(country).strip()
    return country_names.get(country, country)


population_df["country_name"] = population_df["country_name"].apply(rename_countries)
fertility_df["country_name"] = fertility_df["country_name"].apply(rename_countries)

#DATA ENRICHEMENT

def get_project_root():

    # IF CI ENVS
    env_root = os.getenv("PROJECT_ROOT")
    if env_root:
        return Path(env_root)

    # IF SCRIPT .PY FILE
    if "__file__" in globals():
        return Path(__file__).resolve().parent

    # IF NOTEBOOK FALLBACK
    return Path.cwd()

BASE_DIR = get_project_root()

file_path = BASE_DIR / "cleaned_datasets" / "country_enrichment.csv"

if not file_path.exists():
    msg = f"Missing file at: {file_path}"

    if not IS_CI:
        print(msg)
        
    raise FileNotFoundError(msg)

country_enrichment = pd.read_csv(file_path)
enrichment_details = country_enrichment[["country_code", "capital", "land_area_km2", "continent"]]
enrichment_details["country_code"] = enrichment_details["country_code"].str.strip().str.upper()

try:
    population_df = population_df.merge(enrichment_details, on=["country_code"], how="left")
    population_df = population_df[["country_name", "country_code", "year_record", "population_count", "land_area_km2", "capital", "continent"]]
    fertility_df = fertility_df[["country_name", "country_code", "year_record", "tfr"]]

except Exception as merge_issue:
    if not IS_CI:
        print(merge_issue) 

    body = f"""The enrichmenet has failed due to.
    
Error:
{merge_issue}
"""

    #EMAIL NOTIFICATION SEND
    send_email(
        service,
        "Countries_Enrichment - Gmail API",
        body
    )

    raise RuntimeError("Country enrichment failed") from merge_issue

#FINAL DATA CLEANING
population_df["capital"] = population_df["capital"].astype("string").str.strip().str.title()
population_df["continent"] = population_df["continent"].astype("string").str.strip().str.title()
population_df["land_area_km2"] = pd.to_numeric(population_df["land_area_km2"], errors="coerce")

population_df["population_count"] = pd.to_numeric(population_df["population_count"], errors="coerce").astype("Int64")
fertility_df["tfr"] = pd.to_numeric(fertility_df["tfr"], errors="coerce")

# PANDAS NAN VALUES HANDLING
def to_pg_value(x):
    if pd.isna(x):
        return None
    if isinstance(x, np.generic):
        return x.item()
    return x


def df_to_pg_records(df):
    return [
        tuple(to_pg_value(x) for x in row)
        for row in df.itertuples(index=False, name=None)
    ]

#QUICK DATA VALIDATION
#if not IS_CI:
#    print("POPULATION DATA")
#    print(population_df.head(10))

#    print("\nFERTILITY DATA:")
#    print(fertility_df.head(5))

In [ ]:
#----------------------------------------------------------------
# DATABASE AUDITING SECTION
#----------------------------------------------------------------

# CURRENT DATABASE SNAPSHOT FOR AUDITING

db_population = pd.read_sql_query(
    """
    SELECT
        country_code,
        year_record,
        population_count
    FROM fact_population
    """,
    connection
)

db_fertility = pd.read_sql_query(
    """
    SELECT
        country_code,
        year_record,
        tfr
    FROM fact_fertility_rate
    """,
    connection
)

db_countries = pd.read_sql_query(
    """
    SELECT
        country_code,
        country_name,
        capital,
        continent,
        land_area_km2
    FROM dim_countries
    """,
    connection
)

# DATA AUDITING AND COMPARISON

def audit_changes(
    before_df,
    after_df,
    key_columns,
    value_columns
):

    before = before_df.copy()
    after = after_df.copy()

    # MERGE BEFORE AND AFTER SNAPSHOTS

    comparison = before.merge(
        after,
        on=key_columns,
        how="outer",
        suffixes=("_before", "_after"),
        indicator=True
    )

    # INITIAL STATUS


    comparison["status"] = np.select(
        [
            comparison["_merge"] == "right_only",
            comparison["_merge"] == "left_only",
            comparison["_merge"] == "both"
        ],
        [
            "INSERTED",
            "REMOVED",
            "EXISTING"
        ],
        default="UNKNOWN"
    )

    # DETECT ACTUAL VALUE CHANGES

    def values_are_different(row):

        for column in value_columns:

            before_value = row[f"{column}_before"]
            after_value = row[f"{column}_after"]

            # Both values are NULL
            if pd.isna(before_value) and pd.isna(after_value):
                continue

            # One value is NULL and the other is not
            if pd.isna(before_value) != pd.isna(after_value):
                return True

            # Values are different
            if before_value != after_value:
                return True

        return False

    # CLASSIFY EXISTING RECORDS

    existing_mask = comparison["_merge"] == "both"

    comparison.loc[existing_mask, "status"] = (
        comparison.loc[existing_mask]
        .apply(
            lambda row:
                "UPDATED"
                if values_are_different(row)
                else "UNCHANGED",
            axis=1
        )
    )

    # REMOVE TECHNICAL MERGE COLUMN

    comparison = comparison.drop(columns=["_merge"])

    return comparison

In [ ]:
#----------------------------------------------------------------
# PDF AUDIT REPORT GENERATION FUNCTION
#----------------------------------------------------------------

def create_audit_pdf(
    population_audit,
    fertility_audit,
    countries_audit,
    total_years,
    population_completeness,
    fertility_completeness,
    population_total_years_with_values,
    fertility_total_years_with_values,
    audit_pdf_path
):

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "AuditTitle",
        parent=styles["Title"],
        alignment=TA_CENTER,
        fontSize=20,
        spaceAfter=10
    )

    subtitle_style = ParagraphStyle(
        "AuditSubtitle",
        parent=styles["Normal"],
        alignment=TA_CENTER,
        fontSize=9,
        spaceAfter=20
    )

    section_style = ParagraphStyle(
        "SectionTitle",
        parent=styles["Heading2"],
        fontSize=14,
        spaceBefore=20,
        spaceAfter=20
    )

    normal_style = ParagraphStyle(
        "AuditNormal",
        parent=styles["Normal"],
        fontSize=9
    )

    # PDF DOCUMENT

    document = SimpleDocTemplate(
        str(audit_pdf_path),
        pagesize=landscape(letter),
        rightMargin=30,
        leftMargin=30,
        topMargin=30,
        bottomMargin=30
    )

    elements = []

    # REPORT HEADER

    elements.append(
        Paragraph(
            "WORLD POPULATION ETL AUDIT REPORT",
            title_style
        )
    )

    elements.append(
        Paragraph(
            f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            subtitle_style
        )
    )

    # SUMMARY TABLE FUNCTION

    def create_summary_table(audit_df):

        counts = audit_df["status"].value_counts()

        data = [
            ["Status", "Records"],
            ["INSERTED", counts.get("INSERTED", 0)],
            ["UPDATED", counts.get("UPDATED", 0)],
            ["UNCHANGED", counts.get("UNCHANGED", 0)],
            ["REMOVED", counts.get("REMOVED", 0)]
        ]

        table = Table(
            data,
            colWidths=[2.5 * inch, 1.5 * inch]
        )

        table.setStyle(
            TableStyle([
                (
                    "BACKGROUND",
                    (0, 0),
                    (-1, 0),
                    colors.HexColor("#1F2937")
                ),
                (
                    "TEXTCOLOR",
                    (0, 0),
                    (-1, 0),
                    colors.white
                ),
                (
                    "FONTNAME",
                    (0, 0),
                    (-1, 0),
                    "Helvetica-Bold"
                ),
                (
                    "GRID",
                    (0, 0),
                    (-1, -1),
                    0.5,
                    colors.grey
                ),
                (
                    "ALIGN",
                    (1, 1),
                    (1, -1),
                    "CENTER"
                ),
                (
                    "ROWBACKGROUNDS",
                    (0, 1),
                    (-1, -1),
                    [
                        colors.white,
                        colors.HexColor("#F3F4F6")
                    ]
                )
            ])
        )

        return table

    # POPULATION AUDIT

    elements.append(
        Paragraph(
            "POPULATION AUDIT",
            section_style
        )
    )

    elements.append(
        create_summary_table(population_audit)
    )

    elements.append(Spacer(1, 75))

    population_changes = population_audit[
        population_audit["status"].isin(
            ["INSERTED", "UPDATED", "REMOVED"]
        )
    ].copy()

    # POPULATION CHANGE MESSAGE

    if population_changes.empty:

        elements.append(
            Paragraph(
                "No population changes detected during this ETL run.",
                normal_style
            )
        )

    else:

        elements.append(
            Paragraph(
                f"{len(population_changes)} population changes detected.",
                normal_style
            )
        )

        elements.append(
            Paragraph(
                "Population Changes",
                styles["Heading3"]
            )
        )

        # POPULATION CHANGE TABLE

        data = [
            [
                "Code",
                "Year",
                "Previous",
                "Current",
                "Delta",
                "Status"
            ]
        ]

        for _, row in population_changes.iterrows():

            data.append([
                row["country_code"],
                row["year_record"],
                (
                    f'{row["population_count_before"]:,}'
                    if pd.notna(row["population_count_before"])
                    else "—"
                ),
                (
                    f'{row["population_count_after"]:,}'
                    if pd.notna(row["population_count_after"])
                    else "—"
                ),
                (
                    f'{row["population_delta"]:+,}'
                    if pd.notna(row["population_delta"])
                    else "—"
                ),
                row["status"]
            ])

        table = Table(
            data,
            repeatRows=1
        )

        table.setStyle(
            TableStyle([
                (
                    "BACKGROUND",
                    (0, 0),
                    (-1, 0),
                    colors.HexColor("#1F2937")
                ),
                (
                    "TEXTCOLOR",
                    (0, 0),
                    (-1, 0),
                    colors.white
                ),
                (
                    "FONTNAME",
                    (0, 0),
                    (-1, 0),
                    "Helvetica-Bold"
                ),
                (
                    "FONTSIZE",
                    (0, 0),
                    (-1, -1),
                    8
                ),
                (
                    "GRID",
                    (0, 0),
                    (-1, -1),
                    0.5,
                    colors.grey
                ),
                (
                    "ALIGN",
                    (1, 1),
                    (-2, -1),
                    "CENTER"
                ),
                (
                    "ROWBACKGROUNDS",
                    (0, 1),
                    (-1, -1),
                    [
                        colors.white,
                        colors.HexColor("#F9FAFB")
                    ]
                )
            ])
        )

        elements.append(table)

    # POPULATION OVERALL COMPLETENESS SUMMARY

    elements.append(Spacer(1, 75))

    population_overall_data = [
        ["Metric", "Count"],
        [
            "Total available years",
            total_years
        ],
        [
            "Total years with population values",
            population_total_years_with_values
        ],
        [
            "Total years missing population values",
            total_years - population_total_years_with_values
        ]
    ]

    population_overall_table = Table(
        population_overall_data,
        colWidths=[3.5 * inch, 1.5 * inch]
    )

    population_overall_table.setStyle(
        TableStyle([
            (
                "BACKGROUND",
                (0, 0),
                (-1, 0),
                colors.HexColor("#1F2937")
            ),
            (
                "TEXTCOLOR",
                (0, 0),
                (-1, 0),
                colors.white
            ),
            (
                "FONTNAME",
                (0, 0),
                (-1, 0),
                "Helvetica-Bold"
            ),
            (
                "GRID",
                (0, 0),
                (-1, -1),
                0.5,
                colors.grey
            ),
            (
                "ALIGN",
                (1, 1),
                (1, -1),
                "CENTER"
            ),
            (
                "ROWBACKGROUNDS",
                (0, 1),
                (-1, -1),
                [
                    colors.white,
                    colors.HexColor("#F9FAFB")
                ]
            )
        ])
    )

    elements.append(population_overall_table)

    # FERTILITY AUDIT

    elements.append(PageBreak())

    elements.append(
        Paragraph(
            "FERTILITY AUDIT",
            section_style
        )
    )

    elements.append(
        create_summary_table(fertility_audit)
    )

    elements.append(Spacer(1, 75))

    fertility_changes = fertility_audit[
        fertility_audit["status"].isin(
            ["INSERTED", "UPDATED", "REMOVED"]
        )
    ].copy()

    # FERTILITY CHANGE MESSAGE

    if fertility_changes.empty:

        elements.append(
            Paragraph(
                "No fertility changes detected during this ETL run.",
                normal_style
            )
        )

    else:

        elements.append(
            Paragraph(
                f"{len(fertility_changes)} fertility changes detected.",
                normal_style
            )
        )

        elements.append(
            Paragraph(
                "Fertility Rate Changes",
                styles["Heading3"]
            )
        )

        # FERTILITY CHANGE TABLE

        data = [
            [
                "Code",
                "Year",
                "Previous",
                "Current",
                "Delta",
                "Status"
            ]
        ]

        for _, row in fertility_changes.iterrows():

            data.append([
                row["country_code"],
                row["year_record"],
                (
                    f'{row["tfr_before"]:.2f}'
                    if pd.notna(row["tfr_before"])
                    else "—"
                ),
                (
                    f'{row["tfr_after"]:.2f}'
                    if pd.notna(row["tfr_after"])
                    else "—"
                ),
                (
                    f'{row["tfr_delta"]:+.2f}'
                    if pd.notna(row["tfr_delta"])
                    else "—"
                ),
                row["status"]
            ])

        table = Table(
            data,
            repeatRows=1
        )

        table.setStyle(
            TableStyle([
                (
                    "BACKGROUND",
                    (0, 0),
                    (-1, 0),
                    colors.HexColor("#1F2937")
                ),
                (
                    "TEXTCOLOR",
                    (0, 0),
                    (-1, 0),
                    colors.white
                ),
                (
                    "FONTNAME",
                    (0, 0),
                    (-1, 0),
                    "Helvetica-Bold"
                ),
                (
                    "FONTSIZE",
                    (0, 0),
                    (-1, -1),
                    8
                ),
                (
                    "GRID",
                    (0, 0),
                    (-1, -1),
                    0.5,
                    colors.grey
                ),
                (
                    "ALIGN",
                    (1, 1),
                    (-2, -1),
                    "CENTER"
                ),
                (
                    "ROWBACKGROUNDS",
                    (0, 1),
                    (-1, -1),
                    [
                        colors.white,
                        colors.HexColor("#F9FAFB")
                    ]
                )
            ])
        )

        elements.append(table)

    # FERTILITY OVERALL COMPLETENESS SUMMARY

    elements.append(Spacer(1, 75))

    fertility_overall_data = [
        ["Metric", "Count"],
        [
            "Total available years",
            total_years
        ],
        [
            "Total years with fertility values",
            fertility_total_years_with_values
        ],
        [
            "Total years missing fertility values",
            total_years - fertility_total_years_with_values
        ]
    ]

    fertility_overall_table = Table(
        fertility_overall_data,
        colWidths=[3.5 * inch, 1.5 * inch]
    )

    fertility_overall_table.setStyle(
        TableStyle([
            (
                "BACKGROUND",
                (0, 0),
                (-1, 0),
                colors.HexColor("#1F2937")
            ),
            (
                "TEXTCOLOR",
                (0, 0),
                (-1, 0),
                colors.white
            ),
            (
                "FONTNAME",
                (0, 0),
                (-1, 0),
                "Helvetica-Bold"
            ),
            (
                "GRID",
                (0, 0),
                (-1, -1),
                0.5,
                colors.grey
            ),
            (
                "ALIGN",
                (1, 1),
                (1, -1),
                "CENTER"
            ),
            (
                "ROWBACKGROUNDS",
                (0, 1),
                (-1, -1),
                [
                    colors.white,
                    colors.HexColor("#F9FAFB")
                ]
            )
        ])
    )

    elements.append(fertility_overall_table)

    # COUNTRIES AUDIT

    elements.append(PageBreak())

    elements.append(
        Paragraph(
            "COUNTRIES AUDIT",
            section_style
        )
    )

    elements.append(
        create_summary_table(countries_audit)
    )

    elements.append(Spacer(1, 75))

    country_changes = countries_audit[
        countries_audit["status"].isin(
            ["INSERTED", "UPDATED", "REMOVED"]
        )
    ].copy()

    # COUNTRY CHANGE MESSAGE

    if country_changes.empty:

        elements.append(
            Paragraph(
                "No country dimension changes detected during this ETL run.",
                normal_style
            )
        )

    else:

        elements.append(
            Paragraph(
                f"{len(country_changes)} country dimension changes detected.",
                normal_style
            )
        )

        elements.append(
            Paragraph(
                "Country Dimension Changes",
                styles["Heading3"]
            )
        )

        # COUNTRY CHANGE TABLE

        data = [
            [
                "Code",
                "Previous Name",
                "Current Name",
                "Previous Capital",
                "Current Capital",
                "Previous Continent",
                "Current Continent",
                "Previous Area",
                "Current Area",
                "Status"
            ]
        ]

        for _, row in country_changes.iterrows():

            data.append([
                row["country_code"],

                (
                    row["country_name_before"]
                    if pd.notna(row["country_name_before"])
                    else "—"
                ),

                (
                    row["country_name_after"]
                    if pd.notna(row["country_name_after"])
                    else "—"
                ),

                (
                    row["capital_before"]
                    if pd.notna(row["capital_before"])
                    else "—"
                ),

                (
                    row["capital_after"]
                    if pd.notna(row["capital_after"])
                    else "—"
                ),

                (
                    row["continent_before"]
                    if pd.notna(row["continent_before"])
                    else "—"
                ),

                (
                    row["continent_after"]
                    if pd.notna(row["continent_after"])
                    else "—"
                ),

                (
                    f'{row["land_area_km2_before"]:,.0f}'
                    if pd.notna(row["land_area_km2_before"])
                    else "—"
                ),

                (
                    f'{row["land_area_km2_after"]:,.0f}'
                    if pd.notna(row["land_area_km2_after"])
                    else "—"
                ),

                row["status"]
            ])

        table = Table(
            data,
            repeatRows=1,
            colWidths=[
                0.55 * inch,
                1.00 * inch,
                1.00 * inch,
                1.00 * inch,
                1.00 * inch,
                0.90 * inch,
                0.90 * inch,
                0.75 * inch,
                0.75 * inch,
                0.65 * inch
            ]
        )

        table.setStyle(
            TableStyle([
                (
                    "BACKGROUND",
                    (0, 0),
                    (-1, 0),
                    colors.HexColor("#1F2937")
                ),
                (
                    "TEXTCOLOR",
                    (0, 0),
                    (-1, 0),
                    colors.white
                ),
                (
                    "FONTNAME",
                    (0, 0),
                    (-1, 0),
                    "Helvetica-Bold"
                ),
                (
                    "FONTSIZE",
                    (0, 0),
                    (-1, -1),
                    6
                ),
                (
                    "GRID",
                    (0, 0),
                    (-1, -1),
                    0.5,
                    colors.grey
                ),
                (
                    "ALIGN",
                    (0, 0),
                    (-1, -1),
                    "CENTER"
                ),
                (
                    "VALIGN",
                    (0, 0),
                    (-1, -1),
                    "MIDDLE"
                ),
                (
                    "ROWBACKGROUNDS",
                    (0, 1),
                    (-1, -1),
                    [
                        colors.white,
                        colors.HexColor("#F9FAFB")
                    ]
                )
            ])
        )

        elements.append(table)

    # DATA COMPLETENESS AUDIT

    elements.append(PageBreak())

    elements.append(
        Paragraph(
            "DATA COMPLETENESS AUDIT",
            section_style
        )
    )

    elements.append(
        Paragraph(
            f"Total unique years available in dim_year: {total_years}",
            normal_style
        )
    )

    elements.append(Spacer(1, 75))

    # POPULATION DATA COMPLETENESS

    elements.append(
        Paragraph(
            "POPULATION DATA COMPLETENESS",
            styles["Heading3"]
        )
    )

    elements.append(Spacer(1, 30))

    if not population_completeness:

        elements.append(
            Paragraph(
                "All countries contain population values for every available year.",
                normal_style
            )
        )

    else:

        data = [
            [
                "Country",
                "Code",
                "Years",
                "Years with values",
                "Years Value Missing"
            ]
        ]

        for row in population_completeness:

            data.append([
                row[0],
                row[1],
                row[2],
                row[3],
                row[4]
            ])

        table = Table(
            data,
            repeatRows=1
        )

        table.setStyle(
            TableStyle([
                (
                    "BACKGROUND",
                    (0, 0),
                    (-1, 0),
                    colors.HexColor("#1F2937")
                ),
                (
                    "TEXTCOLOR",
                    (0, 0),
                    (-1, 0),
                    colors.white
                ),
                (
                    "FONTNAME",
                    (0, 0),
                    (-1, 0),
                    "Helvetica-Bold"
                ),
                (
                    "GRID",
                    (0, 0),
                    (-1, -1),
                    0.5,
                    colors.grey
                ),
                (
                    "ALIGN",
                    (1, 1),
                    (-1, -1),
                    "CENTER"
                ),
                (
                    "ROWBACKGROUNDS",
                    (0, 1),
                    (-1, -1),
                    [
                        colors.white,
                        colors.HexColor("#F9FAFB")
                    ]
                )
            ])
        )

        elements.append(table)

    # FERTILITY DATA COMPLETENESS

    elements.append(PageBreak())

    elements.append(
        Paragraph(
            "FERTILITY DATA COMPLETENESS",
            styles["Heading3"]
        )
    )

    elements.append(Spacer(1, 30))

    if not fertility_completeness:

        elements.append(
            Paragraph(
                "All countries contain fertility values for every available year.",
                normal_style
            )
        )

    else:

        data = [
            [
                "Country",
                "Code",
                "Years",
                "Years with values",
                "Years Value Missing"
            ]
        ]

        for row in fertility_completeness:

            data.append([
                row[0],
                row[1],
                row[2],
                row[3],
                row[4]
            ])

        table = Table(
            data,
            repeatRows=1
        )

        table.setStyle(
            TableStyle([
                (
                    "BACKGROUND",
                    (0, 0),
                    (-1, 0),
                    colors.HexColor("#1F2937")
                ),
                (
                    "TEXTCOLOR",
                    (0, 0),
                    (-1, 0),
                    colors.white
                ),
                (
                    "FONTNAME",
                    (0, 0),
                    (-1, 0),
                    "Helvetica-Bold"
                ),
                (
                    "GRID",
                    (0, 0),
                    (-1, -1),
                    0.5,
                    colors.grey
                ),
                (
                    "ALIGN",
                    (1, 1),
                    (-1, -1),
                    "CENTER"
                ),
                (
                    "ROWBACKGROUNDS",
                    (0, 1),
                    (-1, -1),
                    [
                        colors.white,
                        colors.HexColor("#F9FAFB")
                    ]
                )
            ])
        )

        elements.append(table)

    # BUILD REPORT

    document.build(elements)

In [ ]:
#----------------------------------------------------------------
# DATABASE DATA PREPARATION SECTION
#----------------------------------------------------------------

#PREPARING DATA FOR DIM_YEAR VALUES INSERTION

dim_year_table = (pd.concat([population_df[["year_record"]],fertility_df[["year_record"]]]).drop_duplicates().dropna(subset=["year_record"]).assign(
        year_record=lambda df:
            pd.to_numeric(
                df["year_record"],
                errors="raise"
            )
    )
)
dim_year_table_values = df_to_pg_records(dim_year_table)

dim_year_insert_query ="""
INSERT INTO dim_year (year_record)
VALUES (%s)
ON CONFLICT (year_record) DO NOTHING;
"""

dim_countries_table = population_df[["country_name", "country_code", "capital", "continent", "land_area_km2"]].drop_duplicates().dropna(subset=["country_code"])
dim_countries_table_values = df_to_pg_records(dim_countries_table)

dim_countries_insert_query = """
INSERT INTO dim_countries (country_name, country_code, capital, continent, land_area_km2)
VALUES (%s, %s, %s, %s, %s)
ON CONFLICT (country_code)
DO UPDATE SET
    country_name = EXCLUDED.country_name,
    capital = EXCLUDED.capital,
    continent = EXCLUDED.continent,
    land_area_km2 = EXCLUDED.land_area_km2
WHERE
    dim_countries.country_name IS DISTINCT FROM EXCLUDED.country_name
    OR dim_countries.capital IS DISTINCT FROM EXCLUDED.capital
    OR dim_countries.continent IS DISTINCT FROM EXCLUDED.continent
    OR dim_countries.land_area_km2 IS DISTINCT FROM EXCLUDED.land_area_km2;
    """

fact_population_table = population_df[["country_code", "year_record", "population_count"]].drop_duplicates(subset=["country_code", "year_record"]).dropna(subset=["year_record", "country_code"])
fact_population_table_values = df_to_pg_records(fact_population_table)

fact_population_insert_query ="""
INSERT INTO fact_population
(country_code, year_record, population_count)
VALUES (%s, %s, %s)

ON CONFLICT (country_code, year_record)

DO UPDATE
SET population_count = EXCLUDED.population_count

WHERE fact_population.population_count
IS DISTINCT FROM EXCLUDED.population_count;
"""

fact_fertility_rate_table = fertility_df[["country_code", "year_record", "tfr"]].drop_duplicates(subset=["country_code", "year_record"]).dropna(subset=["year_record", "country_code"])
fact_fertility_rate_table_values = df_to_pg_records(fact_fertility_rate_table)

fact_fertility_rate_insert_query ="""
INSERT INTO fact_fertility_rate
(country_code, year_record, tfr)
VALUES (%s, %s, %s)

ON CONFLICT (country_code, year_record)

DO UPDATE
SET tfr = EXCLUDED.tfr

WHERE fact_fertility_rate.tfr
IS DISTINCT FROM EXCLUDED.tfr;"""

In [ ]:
#----------------------------------------------------------------
# DATABASE DATA INSERTION SECTION
#----------------------------------------------------------------

try:
   # LOAD DIM_YEAR
    execute_batch(cur,dim_year_insert_query, dim_year_table_values, page_size=1000)

    # LOAD DIM_COUNTRIES
    execute_batch(cur, dim_countries_insert_query, dim_countries_table_values, page_size=1000)
    
    #LOAD FACT_POPULATION 
    execute_batch(cur, fact_population_insert_query, fact_population_table_values, page_size=1000)

    #LOAD FACT_FERILTITY_RATE
    execute_batch(cur, fact_fertility_rate_insert_query, fact_fertility_rate_table_values, page_size=1000)

    # AFTER DB SNAPSHOT FOR AUDITING

    db_population_after = pd.read_sql_query(
        """
        SELECT
            country_code,
            year_record,
            population_count
        FROM fact_population
        """,
        connection
    )

    db_fertility_after = pd.read_sql_query(
        """
        SELECT
            country_code,
            year_record,
            tfr
        FROM fact_fertility_rate
        """,
        connection
    )

    db_countries_after = pd.read_sql_query(
        """
        SELECT
            country_code,
            country_name,
            capital,
            continent,
            land_area_km2
        FROM dim_countries
        """,
        connection
    )

    population_audit = audit_changes(
        before_df = db_population,
        after_df = db_population_after,
        key_columns = ["country_code", "year_record"],
        value_columns = ["population_count"]
    )

    fertility_audit = audit_changes(
        before_df = db_fertility,
        after_df = db_fertility_after,
        key_columns = ["country_code", "year_record"],
        value_columns = ["tfr"]
    )

    countries_audit = audit_changes(
        before_df = db_countries,
        after_df = db_countries_after,
        key_columns = ["country_code"],
        value_columns = ["country_name", "capital", "continent", "land_area_km2"]
    )

# CALCULATE DELTAS FOR POPULATION AND FERTILITY RATE

    population_audit["population_delta"] = ( population_audit["population_count_after"
    ] - population_audit["population_count_before"]
    )

    fertility_audit["tfr_delta"] = (fertility_audit["tfr_after"
    ] -fertility_audit["tfr_before"]
    )

    population_changes = population_audit[
        population_audit["status"].isin(["INSERTED", "UPDATED", "REMOVED"])
        ]

    fertility_changes = fertility_audit[
        fertility_audit["status"].isin(["INSERTED", "UPDATED", "REMOVED"])
        ]

    country_changes = countries_audit[
        countries_audit["status"].isin(["INSERTED", "UPDATED", "REMOVED"])
    ]

    #POPULATION TABLE AUDIT SUMMARY
    population_counts = population_audit["status"].value_counts()

    population_inserted = population_counts.get("INSERTED", 0)
    population_updated = population_counts.get("UPDATED", 0)
    population_unchanged = population_counts.get("UNCHANGED", 0)
    population_removed = population_counts.get("REMOVED", 0)

    #FERILITY TABLE AUDIT SUMMARY
    fertility_counts = fertility_audit["status"].value_counts()

    fertility_inserted = fertility_counts.get("INSERTED", 0)
    fertility_updated = fertility_counts.get("UPDATED", 0)
    fertility_unchanged = fertility_counts.get("UNCHANGED", 0)
    fertility_removed = fertility_counts.get("REMOVED", 0)


    #COUNTRIES TABLE AUDIT SUMMARY
    country_counts = countries_audit["status"].value_counts()

    country_inserted = country_counts.get("INSERTED", 0)
    country_updated = country_counts.get("UPDATED", 0)
    country_unchanged = country_counts.get("UNCHANGED", 0)
    country_removed = country_counts.get("REMOVED", 0)

#------------------------------------------------
# EMAIL REPORT GENERATION AND SENDING
#------------------------------------------------

# TOTAL AVAILABLE YEARS
    cur.execute("""
        SELECT COUNT(*) 
        FROM dim_year
    """)

    total_years = cur.fetchone()[0]


    # POPULATION COMPLETENESS

    cur.execute("""
    SELECT
        dc.country_name AS "Country",
        dc.country_code AS "Code",

        -- Total years available in dim_year
        COUNT(DISTINCT dy.year_record) AS "Years",

        -- Years where this country actually has a population value
        COUNT(
            DISTINCT CASE
                WHEN fp.population_count IS NOT NULL
                THEN dy.year_record
            END
        ) AS "Years with values",

        -- Years missing a population value
        COUNT(DISTINCT dy.year_record)
        -
        COUNT(
            DISTINCT CASE
                WHEN fp.population_count IS NOT NULL
                THEN dy.year_record
            END
        ) AS "Years Value Missing"

    FROM dim_countries AS dc

    CROSS JOIN dim_year AS dy

    LEFT JOIN fact_population AS fp
        ON dc.country_code = fp.country_code
        AND dy.year_record = fp.year_record

    GROUP BY
        dc.country_name,
        dc.country_code

    HAVING
        COUNT(DISTINCT dy.year_record)
        !=
        COUNT(
            DISTINCT CASE
                WHEN fp.population_count IS NOT NULL
                THEN dy.year_record
            END
        )

    ORDER BY
        dc.country_name;
""")

    population_completeness = cur.fetchall()

# TOTAL YEARS AVAILABLE IN THE DIMENSION
    population_total_years = total_years

# TOTAL DISTINCT YEARS WITH AT LEAST ONE POPULATION VALUE
    cur.execute("""
    SELECT COUNT(DISTINCT year_record)
    FROM fact_population
    WHERE population_count IS NOT NULL
    """)

    population_total_years_with_values = cur.fetchone()[0]

# FERTILITY COMPLETENESS

    cur.execute("""
    SELECT
        dc.country_name AS "Country",
        dc.country_code AS "Code",

        -- Total years available in dim_year
        COUNT(DISTINCT dy.year_record) AS "Years",

        -- Years where this country actually has a fertility value
        COUNT(
            DISTINCT CASE
                WHEN ff.tfr IS NOT NULL
                THEN dy.year_record
            END
        ) AS "Years with values",

        -- Years missing a fertility value
        COUNT(DISTINCT dy.year_record)
        -
        COUNT(
            DISTINCT CASE
                WHEN ff.tfr IS NOT NULL
                THEN dy.year_record
            END
        ) AS "Years Value Missing"

    FROM dim_countries AS dc

    CROSS JOIN dim_year AS dy

    LEFT JOIN fact_fertility_rate AS ff
        ON dc.country_code = ff.country_code
        AND dy.year_record = ff.year_record

    GROUP BY
        dc.country_name,
        dc.country_code

    HAVING
        COUNT(DISTINCT dy.year_record)
        !=
        COUNT(
            DISTINCT CASE
                WHEN ff.tfr IS NOT NULL
                THEN dy.year_record
            END
        )

    ORDER BY
        dc.country_name;
""")

    fertility_completeness = cur.fetchall()

# TOTAL DISTINCT YEARS WITH AT LEAST ONE FERTILITY VALUE
    cur.execute("""
    SELECT COUNT(DISTINCT year_record)
    FROM fact_fertility_rate
    WHERE tfr IS NOT NULL
""")

    fertility_total_years_with_values = cur.fetchone()[0]

# GENERATE AUDIT REPORT

    audit_pdf_path = BASE_DIR / "audit_report.pdf"

    create_audit_pdf(
    population_audit,
    fertility_audit,
    countries_audit,
    total_years,
    population_completeness,
    fertility_completeness,
    population_total_years_with_values,
    fertility_total_years_with_values,
    audit_pdf_path
)   

    # EMAIL NOTIFICATION OF THE SUCCESSFUL INSERTION WiTH THE CONFIRMATION OF THE NUMBER OF VALUES IN THE TABLE
    body = f"The ETL process has completed successfully. Here's your audit report" 

    #EMAIL NOTIFICATION SEND
    send_email(
        service,
        "SUCCESSFUL DATABASE INSERTION",
        body,
        attachment_path=audit_pdf_path
    )

#COMMIT THE INSERTION
    connection.commit()

    if not IS_CI:
        print("SUCCESSFUL DATABASE INSERTION. EMAIL HAS BEEN SENT")

except Exception as e:
    # ROLLBACK TRANSACTION AND SEND FAILURE NOTIFICATION
    connection.rollback()

    body = (
    f"""Failed insertions due to error message:

{e}"""
    )

    #EMAIL NOTIFICATION SEND
    send_email(
        service,
        "FAILED VALUES INSERTION",
        body
    )

    if not IS_CI:
        print(f"""Error occurred: {e}. email sent""")
    raise e
    
    # AFTER THE SCRIPT RUNS SUCCESSFULLY, CLOSE THE DB CONNECTION
finally:

    if cur is not None:
        cur.close()

    if connection is not None:
        connection.close()

    if audit_pdf_path.exists():
        try:
            os.remove(audit_pdf_path)
        except OSError as pdf_cleanup_error:
            if not IS_CI:
                print(f"Warning: could not remove audit PDF: {pdf_cleanup_error}")